# 01 - Simulate PD-Centric Quarterly Panel

This notebook creates a clean synthetic quarterly panel for use in testing PTO, black-box, and DFL approaches.

- the latent credit-risk process is a true quarterly PD process, `PD_quarterly_true`;
- annual PD is derived as `PD_annual_true = 1 - (1 - PD_quarterly_true) ** 4` for Basel IRB;
- realised quarterly default and charge-off rates are sampled conditional on `PD_quarterly_true`;
- the Vasicek asset correlation and Basel IRB capital use `PD_annual_true`;
- oracle allocations use realised quarterly charge-offs and a linear shortfall penalty;
- a Bayes-oracle allocation maximises expected utility under the true Vasicek loss distribution given true PD, so downstream regret can be decomposed into estimation error and irreducible loss noise;
- timing convention for downstream models: predict quarter t+1 from features dated t (macro observables, c_t, c_lag1). By construction of the simulation, x_t and c_t are observable at decision time - there is no reporting lag in the synthetic world.

## Frequency Convention

The empirical FRED `charge_off_rate` series is observed quarterly but reported at an annualised rate. The allocation problem is quarterly, so the simulated realised loss `c` is a quarterly charge-off rate.

The main calibration target is therefore quarterly. The historical annualised charge-off series is converted to a quarterly charge-off proxy and then to a quarterly PD proxy:

```python
hist_c_quarterly = hist_c_annualised / 4
hist_pd_quarterly = hist_c_quarterly / LGD
hist_pd_annual = 1 - (1 - hist_pd_quarterly) ** 4
```

The simulated process starts with `PD_quarterly_true`. Annual PD is derived from it for Basel IRB:

```python
PD_annual_true = 1 - (1 - PD_quarterly_true) ** 4
```

## Part 1 - Imports, paths, and empirical target data

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import norm

pd.set_option("display.float_format", "{:.6f}".format)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks" and PROJECT_ROOT.parent.name == "simulation":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
FIGURE_DIR = PROJECT_ROOT / "figures" / "simulation"

CSV_PATH = DATA_DIR / "simulated_pd_panel.csv"
METADATA_PATH = DATA_DIR / "simulated_pd_panel_metadata.json"
HIST_CHARGE_OFF_PATH = PROCESSED_DATA_DIR / "fred_loan_return_aligned.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Historical charge-off file: {HIST_CHARGE_OFF_PATH}")
print(f"CSV output: {CSV_PATH}")
print(f"Figure directory: {FIGURE_DIR}")

Project root: /Users/hannahokeeffe/Documents/Imperial/DFL_New
Historical charge-off file: /Users/hannahokeeffe/Documents/Imperial/DFL_New/data/processed/fred_loan_return_aligned.csv
CSV output: /Users/hannahokeeffe/Documents/Imperial/DFL_New/data/simulated_pd_panel.csv
Figure directory: /Users/hannahokeeffe/Documents/Imperial/DFL_New/figures/simulation


## Simulation calibration

Change this cell to recalibrate the panel. The key quarterly PD calibration controls are `beta0`, `beta1`, `pd_min`, and `pd_max`; the key allocation controls are `ell` and `lambda_shortfall`.

`AUTO_CALIBRATE_PD = True` searches over `beta0` and `beta1` to match the historical quarterly PD proxy mean and p95. Set it to `False` to use the hand-set beta values.

In [4]:
# Baseline sample and random seed
T = 1000
SEED = 42

# Credit and allocation constants
LGD = 0.45
K1 = 1.0
ell = 10  # leverage; the empirical PTO notebook's LEVERAGE is set to match this
Y_BAR = 0.0168
lambda_shortfall = 10.0
M = 2.5
alpha_grid = np.linspace(0, 1, 101)

# Numerical clipping for inverse-normal and log calculations
PD_CLIP_LOW = 1e-6
PD_CLIP_HIGH = 0.99

# Quarterly PD calibration bounds. These are one-quarter PDs, not annual PDs.
pd_min = 0.001
pd_max = 0.014
beta0 = -2.10
beta1 = 1.10
beta2 = 0.25
AUTO_CALIBRATE_PD = True

# Vasicek draw scaling. RHO_SCALE = 1.0 uses the Basel annual-horizon asset correlation
# unchanged for the quarterly draw. Assumption: this makes realised quarterly losses
# heavier-tailed than the smooth, diversified FRED aggregate (see Table C); set < 1 to
# shrink the systematic loading of the quarterly draw if closer realised tails are wanted.
RHO_SCALE = 1.0

# Basel corporate PD floor (0.03%), matching the empirical PTO notebook. A no-op at this
# calibration (annual PD >= ~0.4%) but kept so the IRB function is identical across notebooks.
APPLY_PD_FLOOR = True
PD_FLOOR = 3e-4

print("Calibration loaded.")

Calibration loaded.


## Part 2 - Historical quarterly calibration targets

In [5]:
hist = pd.read_csv(HIST_CHARGE_OFF_PATH)
hist_c_annualised_raw = hist["charge_off_rate"].dropna().astype(float)

# In this project the FRED annualised rate is stored in percent units when values
# exceed 1. Convert to decimal rates before applying frequency transformations.
charge_off_scale = 100.0 if hist_c_annualised_raw.max() > 1 else 1.0
hist_c_annualised = hist_c_annualised_raw / charge_off_scale

# Convert the annualised empirical charge-off rate to the quarterly decision frequency.
hist_c_quarterly = hist_c_annualised / 4
hist_pd_quarterly = (hist_c_quarterly / LGD).clip(PD_CLIP_LOW, PD_CLIP_HIGH)
hist_pd_annual = 1 - (1 - hist_pd_quarterly) ** 4

print(f"Historical charge-off scale divisor: {charge_off_scale:g}")
print("Historical quarterly charge-off target, decimal units:")
display(hist_c_quarterly.describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]).to_frame("hist_c_quarterly"))
print("Historical quarterly PD proxy, decimal units:")
display(hist_pd_quarterly.describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]).to_frame("hist_pd_quarterly"))
print("Historical annual PD proxy for IRB comparison, decimal units:")
display(hist_pd_annual.describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]).to_frame("hist_pd_annual"))

Historical charge-off scale divisor: 100
Historical quarterly charge-off target, decimal units:


,hist_c_quarterly
count,165.000000
mean,0.001889
std,0.001433
min,0.000300
50%,0.001375
75%,0.003000
90%,0.003820
95%,0.004710
99%,0.005819
max,0.006425


Historical quarterly PD proxy, decimal units:


,hist_pd_quarterly
count,165.000000
mean,0.004197
std,0.003185
min,0.000667
50%,0.003056
75%,0.006667
90%,0.008489
95%,0.010467
99%,0.012931
max,0.014278


Historical annual PD proxy for IRB comparison, decimal units:


,hist_pd_annual
count,165.000000
mean,0.016623
std,0.012519
min,0.002664
50%,0.012166
75%,0.026401
90%,0.033525
95%,0.041214
99%,0.050729
max,0.055900


## Part 3 - Basel helper functions

In [6]:
def clip_pd(pd, low=PD_CLIP_LOW, high=PD_CLIP_HIGH):
    """Clip PD values to avoid numerical issues in log and inverse-normal terms."""
    return np.clip(np.asarray(pd, dtype=float), low, high)


def sigmoid(x):
    """Logistic transform used to map latent stress into bounded PDs."""
    return 1 / (1 + np.exp(-x))


def basel_R(pd_annual):
    """Corporate Basel asset correlation using annual PD."""
    pd_annual = clip_pd(pd_annual)
    exp_term = (1 - np.exp(-50 * pd_annual)) / (1 - np.exp(-50))
    return 0.12 * exp_term + 0.24 * (1 - exp_term)


def maturity_adjustment(pd_annual, M=2.5):
    """Basel maturity adjustment for corporate exposures, using annual PD."""
    pd_annual = clip_pd(pd_annual)
    b = (0.11852 - 0.05478 * np.log(pd_annual)) ** 2
    return (1 + (M - 2.5) * b) / (1 - 1.5 * b)


def k_irb(pd_annual, lgd=0.45, M=2.5, apply_floor=None):
    """Basel IRB capital requirement per unit exposure using annual PD.

    Matches the empirical PTO notebook: corporate PD floor applied before the
    correlation/maturity terms, and the requirement is floored at zero.
    """
    if apply_floor is None:
        apply_floor = APPLY_PD_FLOOR
    pd_annual = clip_pd(pd_annual)
    if apply_floor:
        pd_annual = np.maximum(pd_annual, PD_FLOOR)
    R = basel_R(pd_annual)
    numerator = norm.ppf(pd_annual) + np.sqrt(R) * norm.ppf(0.999)
    denominator = np.sqrt(1 - R)
    unexpected_loss = norm.cdf(numerator / denominator) - pd_annual
    return np.maximum(lgd * unexpected_loss * maturity_adjustment(pd_annual, M=M), 0.0)

## Part 5 - Simulate latent macro state

In [8]:
def simulate_latent_state(T=1000, phi=0.95, sigma=0.35, seed=42):
    """Simulate persistent latent stress and a second persistent component."""
    rng = np.random.default_rng(seed)
    state = np.zeros(T)
    cycle = np.zeros(T)

    for t in range(1, T):
        state[t] = phi * state[t - 1] + rng.normal(0, sigma)
        cycle[t] = 0.75 * cycle[t - 1] + rng.normal(0, 0.45)

    state = (state - state.mean()) / state.std(ddof=0)
    cycle = (cycle - cycle.mean()) / cycle.std(ddof=0)
    return pd.DataFrame({"latent_state": state, "latent_cycle": cycle})


latent = simulate_latent_state(T=T, seed=SEED)
latent.head()

,latent_state,latent_cycle
0,0.423821,0.115482
1,0.520914,-0.610616
2,0.755177,0.227594
3,0.116947,-0.709592
4,0.173025,-0.724119


## Part 7 - Generate and calibrate true quarterly PD path

## PD calibration experiment: Specification A vs B vs C

This notebook compares the three quarterly true-PD calibration objectives before any RHO_SCALE calibration.


In [ ]:
from scipy.optimize import differential_evolution

PD_MOMENTS = ["mean", "std", "p50", "p75", "p90", "p95", "p99", "min", "max"]
PD_CALIBRATION_BOUNDS = [(-6.0, 0.0), (0.0, 4.0), (-2.0, 2.0)]
PD_CALIBRATION_SEED = 2718

def quarterly_pd_from_latent(latent_df, beta0, beta1, beta2, pd_min, pd_max):
    s = latent_df["latent_state"].to_numpy()
    q = latent_df["latent_cycle"].to_numpy()
    pd_index = beta0 + beta1 * s + beta2 * q
    return np.clip(pd_min + sigmoid(pd_index) * (pd_max - pd_min), PD_CLIP_LOW, PD_CLIP_HIGH)

def pd_moments(values):
    x = np.asarray(values, dtype=float)
    return pd.Series({
        "mean": np.mean(x), "std": np.std(x, ddof=1), "p50": np.quantile(x,.50),
        "p75": np.quantile(x,.75), "p90": np.quantile(x,.90), "p95": np.quantile(x,.95),
        "p99": np.quantile(x,.99), "min": np.min(x), "max": np.max(x)
    })

hist_pd_targets = pd_moments(hist_pd_quarterly)

def pd_beta_objective(params, target_moments, objective_moments):
    simulated = pd_moments(quarterly_pd_from_latent(latent, *params, pd_min, pd_max))
    return float(sum(((simulated[m] - target_moments[m]) / max(abs(target_moments[m]), 1e-12)) ** 2 for m in objective_moments))

PD_SPECIFICATIONS = {
    "A_mean_p95": ["mean", "p95"],
    "B_mean_std_p90_p95": ["mean", "std", "p90", "p95"],
    "C_mean_std_p75_p90_p95_p99": ["mean", "std", "p75", "p90", "p95", "p99"],
}

def fit_beta_specification(name, objective_moments):
    result = differential_evolution(
        lambda params: pd_beta_objective(params, hist_pd_targets, objective_moments),
        bounds=PD_CALIBRATION_BOUNDS, seed=PD_CALIBRATION_SEED, polish=True,
        tol=1e-8, maxiter=120, popsize=10, workers=1
    )
    path = quarterly_pd_from_latent(latent, *result.x, pd_min, pd_max)
    simulated = pd_moments(path)
    table = pd.DataFrame({"historical":hist_pd_targets, "simulated":simulated})
    table["absolute_error"] = table["simulated"] - table["historical"]
    table["percentage_error"] = 100 * table["absolute_error"] / table["historical"].abs().clip(lower=1e-12)
    return {"name":name, "objective_moments":objective_moments, "params":result.x,
            "objective":float(result.fun), "path":path, "table":table}

pd_specification_results = {name: fit_beta_specification(name, moments) for name, moments in PD_SPECIFICATIONS.items()}
print("Historical quarterly implied-PD targets:")
display(hist_pd_targets.to_frame("historical_PD_quarterly"))
print("\nBeta calibration specifications:")
for name, fit in pd_specification_results.items():
    b0,b1,b2 = fit["params"]
    print(f"{name}: beta0={b0:.6f}, beta1={b1:.6f}, beta2={b2:.6f}, objective={fit['objective']:.8g}")
    display(fit["table"])

# Explicit selection: Specification B is the baseline passed to rho calibration.
SELECTED_PD_SPECIFICATION = "B_mean_std_p90_p95"
selected_beta_fit = pd_specification_results[SELECTED_PD_SPECIFICATION]
beta0_fixed, beta1_fixed, beta2_fixed = map(float, selected_beta_fit["params"])
PD_quarterly_true_fixed = selected_beta_fit["path"]
PD_annual_true_fixed = 1 - (1 - PD_quarterly_true_fixed) ** 4

print(f"\nSelected {SELECTED_PD_SPECIFICATION} and fixed beta values for rho calibration:")
print(f"beta0_fixed={beta0_fixed:.6f}, beta1_fixed={beta1_fixed:.6f}, beta2_fixed={beta2_fixed:.6f}")

## Specification comparison plots


In [ ]:
# Specification comparison plots
for name, fit in pd_specification_results.items():
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
    axes[0].hist(hist_pd_quarterly, bins=30, density=True, alpha=.55, label="Historical")
    axes[0].hist(fit["path"], bins=30, density=True, alpha=.5, label="Simulated")
    axes[0].set_title(f"{name}: PD distributions")
    axes[0].set_xlabel("Quarterly implied PD")
    axes[0].legend()
    qs=np.linspace(.01,.99,99)
    axes[1].plot(qs, hist_pd_quarterly.quantile(qs), label="Historical")
    axes[1].plot(qs, pd.Series(fit["path"]).quantile(qs), label="Simulated")
    axes[1].set_title("Quantile comparison")
    axes[1].set_xlabel("Quantile")
    axes[1].legend()
    axes[2].plot(fit["path"], lw=1.0, label="Calibrated true quarterly PD")
    axes[2].set_title("Calibrated PD path")
    axes[2].set_xlabel("Quarter")
    axes[2].legend()
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].scatter(latent["latent_state"], fit["path"], s=10, alpha=.45)
    axes[0].set(xlabel="s_t", ylabel="True quarterly PD", title=f"{name}: PD vs s_t")
    axes[1].scatter(latent["latent_cycle"], fit["path"], s=10, alpha=.45)
    axes[1].set(xlabel="q_t", ylabel="True quarterly PD", title=f"{name}: PD vs q_t")
    plt.tight_layout()
    plt.show()
